# 03 调参与交付

对应教程章节：[06-调参方法](../06-调参方法.md)、[07-排错手册](../07-排错手册.md)、[08-进阶与交付](../08-进阶与交付.md)

**本 notebook 的终点**：把指标往上推一档，然后规范地把作业交出去。

## 0. 环境自检

In [ ]:
# ===== 环境自检与路径准备：每个 notebook 都先运行这一格 =====
import os, sys, pathlib, subprocess

# 0) 记下 notebook 启动时所在目录（后面用它找配套脚本），再切到作业根目录
START_DIR = pathlib.Path.cwd()

# 1) 缓存目录指到可写位置，避免 matplotlib / ultralytics 的权限报错
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplcache")
os.environ.setdefault("YOLO_CONFIG_DIR", "/tmp/ultralytics")

# 2) 定位作业根目录：向上查找含 data/ 或 datasets/ 或 codebase/ 的目录
def find_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for cand in (p, *p.parents):
        if any((cand / d).is_dir() for d in ("data", "datasets", "codebase")):
            return cand
    return p

ROOT = find_root()
os.chdir(ROOT)
print("作业根目录:", ROOT)

# 3) 与配套脚本一致：优先使用作业自带的 ultralytics 源码
CB = ROOT / "codebase" / "ultralytics-YOLO26"
if CB.is_dir() and str(CB) not in sys.path:
    sys.path.insert(0, str(CB))

# 4) 依赖与 GPU 自检
missing = []
for mod, name in (("torch", "PyTorch"), ("ultralytics", "ultralytics"),
                  ("PIL", "Pillow"), ("matplotlib", "matplotlib")):
    try:
        __import__(mod)
    except ImportError:
        missing.append(name)
if missing:
    print("❌ 缺少依赖:", missing)
    print("   请在 ai-env 里执行：conda activate ai-env && pip install ipykernel matplotlib")
else:
    import torch, ultralytics
    print("✅ 依赖齐全")
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "（无 GPU，训练会非常慢）"
    print("CUDA 可用:", torch.cuda.is_available(), "|", gpu)
    print("ultralytics:", ultralytics.__version__, "->", ultralytics.__file__)


In [ ]:
# ===== 工具函数：找脚本 / 运行脚本 / 显示图片 =====
# 画图后端：在 Jupyter / IPython 里必须用 inline，否则 plt.show() 不会把图显示出来
# （纯脚本环境没有 IPython，就退回 Agg —— 不弹窗、只保存文件）
try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def script(name):
    """定位配套脚本，兼容三种常见目录布局：

    1. <作业根>/code/train.py      —— 教程默认布局（把 code/ 拷进作业根）
    2. <作业根>/train.py           —— 把脚本直接放在作业根
    3. <notebook 同级的 code/>     —— 直接在教程文件夹里运行 notebook
    """
    cands = [ROOT / "code" / name, ROOT / name,
             START_DIR.parent / "code" / name, START_DIR / "code" / name]
    for c in cands:
        if c.exists():
            return str(c)
    raise FileNotFoundError(
        f"找不到 {name}。请把教程的 code/ 目录复制到作业根目录（与 data/ 同级），"
        f"或把脚本直接放在作业根目录。已查找：{[str(c) for c in cands]}")

def run(*args, check=True):
    """用当前内核的 Python 运行配套脚本，输出直接显示在下面。"""
    cmd = [sys.executable, *[str(a) for a in args]]
    print("$", " ".join(cmd), "\n")
    return subprocess.run(cmd, check=check)

def show(path, width=800, title=None):
    """在 notebook 里显示一张本地图片。

    图片文件直接用 IPython.display 显示（不经过 matplotlib，最稳）；
    不在 Jupyter 里时退回 PIL + matplotlib，至少不会报错。
    """
    path = pathlib.Path(path)
    if not path.exists():
        print("⚠️ 文件不存在:", path, "（先运行生成它的那一格）")
        return
    if title:
        print(title)
    try:
        from IPython.display import Image as _IPyImage, display
        display(_IPyImage(filename=str(path), width=width))
        return
    except ImportError:
        pass
    img = Image.open(path)
    w, h = img.size
    fig, ax = plt.subplots(figsize=(8, 8 * h / w))
    ax.imshow(img); ax.axis("off")
    if title:
        ax.set_title(title)
    plt.show()


## 1. 先按症状定位，再动手

调参最重要的一步不是改参数，而是**先看指标结构判断瓶颈在哪**：

| 症状 | 说明 | 优先动作 |
|---|---|---|
| mAP50 高、mAP75 / mAP50-95 低 | 定位精度不足（小目标、分辨率不够） | **提高 imgsz** |
| mAP50 和 mAP50-95 都低 | 数据/标注问题，或训练严重不足 | 回查数据，别急着调参 |
| 训练损失降、验证损失升 | 过拟合 | 靠早停 + `best.pt`；增加数据或增强 |
| 指标震荡剧烈 | 学习率偏大 / batch 偏小 / 划分不均 | 降学习率；检查划分 |
| 训练慢、GPU 利用率低 | 数据读取瓶颈 | 加 `--cache`，不要加 workers |

三条铁律：**一次只改一个变量**、**只在 val 上调参**、**test 只用一次**。
优先级：**补数据 > 提高分辨率 > 换更大模型 > 调超参**。

## 2. 先测单轮耗时，再定 epoch 数

时间不够时最容易翻车的做法是"先跑起来再说"。用 20% 数据跑 2 轮量出耗时，再反推 epoch。

In [ ]:
run(script("train.py"), "--name", "timing", "--epochs", "2", "--fraction", "0.2",
    "--imgsz", "640", "--batch", "8", "--workers", "2", "--device", "0")

In [ ]:
import csv

timing = ROOT / "runs/timing/results.csv"
base = ROOT / "runs/baseline_n_640/results.csv"

if timing.exists():
    rows = list(csv.DictReader(open(timing)))
    print(f"20% 数据下单轮约 {float(rows[-1]['time'])/len(rows):.1f} 秒（全量约为它的 5 倍）")
else:
    print("⚠️ 还没找到 runs/timing/results.csv —— 先运行上面那格计时训练。")

if base.exists():
    b = list(csv.DictReader(open(base)))
    full = float(b[-1]["time"]) / len(b)
    print(f"baseline 全量实测：{full:.1f} 秒/轮")
    print("\n按单轮耗时选 epoch 数：")
    for lo, hi, ep in ((0, 30, "120~200"), (30, 60, "80"), (60, 120, "50"), (120, 10**9, "40，并考虑降 imgsz")):
        mark = "   ← 你的情况" if lo <= full < hi else ""
        print(f"  {lo:>3}~{'∞' if hi > 10**8 else hi:>3} 秒/轮 -> {ep}{mark}")
else:
    print("⚠️ 还没找到 baseline 的逐轮数据 —— 先运行 notebook 02 的 baseline 训练。")

## 3. 分辨率实验（本次唯一带来实质提升的变量）

思路：baseline 的 mAP50 已经 0.94，但 mAP50-95 只有 0.63——"能找到目标，但框不够贴合"。
数据里 HUD 截图的小目标只有几十像素，640 输入下采样后信息不足，所以提高分辨率应该有效。

**只改 `imgsz` 一个变量**（batch 从 8 降到 4 是显存所迫，不算独立变量）。

⏱ 约 55 分钟（22 秒/轮 × 150 轮）。

In [ ]:
run(script("train.py"), "--name", "exp_imgsz960", "--epochs", "150", "--imgsz", "960",
    "--batch", "4", "--workers", "2", "--device", "0")

In [ ]:
# 在 test 上评估一次（imgsz 必须与训练一致，否则白掉分）
run(script("eval.py"), "--weights", "runs/exp_imgsz960/weights/best.pt",
    "--split", "test", "--imgsz", "960", "--batch", "4", "--device", "0")

### 3.1 三组数字放一起看

In [ ]:
import csv

summary_path = ROOT / "runs/summary.csv"
if not summary_path.exists():
    print("⚠️ 还没找到 runs/summary.csv —— 先运行上面的评估单元（eval.py 会创建并累加这张表）。")
else:
    rows = list(csv.DictReader(open(summary_path)))
    cols = ["run", "split", "imgsz", "precision", "recall", "mAP50", "mAP75", "mAP50-95"]
    print(" | ".join(f"{c:>10}" for c in cols)); print("-" * 100)
    for r in rows:
        print(" | ".join(f"{str(r.get(c, '')):>10}" for c in cols))

    base = [r for r in rows if r["run"] == "baseline_n_640" and r["split"] == "test"]
    new = [r for r in rows if r["run"] == "exp_imgsz960" and r["split"] == "test"]
    if base and new:
        b, n = base[-1], new[-1]
        print("\n相对 baseline 的变化：")
        for k in ("mAP50", "mAP75", "mAP50-95"):
            bv, nv = float(b[k]), float(n[k])
            print(f"  {k:<9} {bv:.4f} -> {nv:.4f}   ({(nv-bv)/bv*100:+.1f}%)")
    else:
        print("\n（还没有 test 结果，或实验名不是 baseline_n_640 / exp_imgsz960）")

**参考结果**：mAP50 +2.1%、**mAP75 +6.3%**、mAP50-95 +5.4%。**提升幅度随 IoU 阈值升高而放大**，
正好印证"瓶颈在定位精度"的判断——也说明如果一开始就无脑加 epoch，得不到这个提升。

## 4. 未标注数据：新手最容易用错的一块

只有一种用法正确：**模型预标注 + 人工核对后并入 train**。
另两种都错：直接配空标签当背景（图里其实有目标，等于教模型"这里不要框"）、放进 val/test（没标签算不出可信指标）。

In [ ]:
run(script("autolabel.py"), "--conf", "0.25", "--device", "0")

In [ ]:
import csv

rows = list(csv.DictReader(open(ROOT / "datasets/rm_addon/autolabel_stats.csv")))
boxes = [int(r["boxes"]) for r in rows]
confs = sorted(float(r["max_conf"]) for r in rows)
n = len(rows)
print(f"处理 {n} 张 | 总框 {sum(boxes)} | 平均 {sum(boxes)/n:.2f} 框/图")
print(f"最高置信度: 中位 {confs[n//2]:.2f} | max {confs[-1]:.2f}")
zero = [r["image"] for r in rows if int(r["boxes"]) == 0]
print(f"零检出 {len(zero)} 张: {zero[:8]}")
print("\n对比：人工标注平均 4.54 框/图。两者接近，说明模型对这批数据的理解是稳定的。")

In [ ]:
import glob
for p in sorted(glob.glob(str(ROOT / "datasets/rm_addon/review/*.jpg")))[:3]:
    show(p, width=800, title=pathlib.Path(p).name)

**必须人工过一遍再并入**。核对重点：框错位置的、该框没框的、同一目标重复框的。

核对完再执行下面两格（构建独立数据集，**原始数据集不动**，不满意随时删目录回滚）：

In [ ]:
run(script("merge_addon.py"), "--dry-run", "--min-conf", "0.5")

In [ ]:
run(script("merge_addon.py"), "--min-conf", "0.5")

合并后生成 `datasets/rm_plus/`，用它训练并在 test 上判定是否真的变好：

```bash
python code/train.py --name exp_addon --data datasets/rm_plus/data.yaml --epochs 150 --imgsz 960 --batch 4
python code/eval.py --weights runs/exp_addon/weights/best.pt --split test --imgsz 960
```

**判定规则要提前定死**（例如"mAP75 超过原模型才采纳，否则保留原模型并把负结果也记下来"）——事后找理由就是自欺欺人。

## 5. 交付

### 5.1 训练表格（自动生成）

In [ ]:
import csv, yaml

summary_path = ROOT / "runs/summary.csv"
if not summary_path.exists():
    print("⚠️ 还没找到 runs/summary.csv —— 先跑评估。")
else:
    summary = list(csv.DictReader(open(summary_path)))
    print("| 编号 | 训练配置（epoch / 学习率 / 优化器 / 尺寸 / batch） | 测试集结果（mAP50 / mAP75） |")
    print("|---|---|---|")
    for i, r in enumerate([x for x in summary if x["split"] == "test"], 1):
        args_p = ROOT / f"runs/{r['run']}/args.yaml"
        if args_p.exists():
            a = yaml.safe_load(open(args_p))
            cfg = (f"{a.get('epochs')} / {a.get('lr0') or 'auto'} / {a.get('optimizer')} / "
                   f"{a.get('imgsz')} / {a.get('batch')}")
        else:
            cfg = "（缺 args.yaml）"
        print(f"| {i} | {cfg} | {r['mAP50']} / {r['mAP75']} |")

### 5.2 打包

命名规则：**压缩包名 = 邮件主题 = PDF 文件名**，权重按作业要求命名（通常是 `<姓名>.pt`）。

In [ ]:
import shutil, zipfile

NAME = "你的姓名"                        # ← 改成你的名字
SUBJECT = "RM第四次培训-班级-姓名"         # ← 改成作业通知要求的格式
weight = ROOT / "runs/exp_imgsz960/weights/best.pt"
pdf = ROOT / f"{SUBJECT}.pdf"             # 先导出 PDF 放到作业根目录

shutil.copy(weight, ROOT / f"{NAME}.pt")
print("权重已复制为:", ROOT / f"{NAME}.pt")

if pdf.exists():
    zip_path = ROOT / f"{SUBJECT}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pdf, pdf.name)
        z.write(ROOT / f"{NAME}.pt", f"{NAME}.pt")
    print("已打包:", zip_path, f"({zip_path.stat().st_size/1048576:.2f} MB)")
    print("包内文件:", zipfile.ZipFile(zip_path).namelist())
else:
    print("⚠️ 还没找到 PDF：", pdf)
    print("   先把文档导出成 PDF（要在文档所在目录导出，否则引用的图片会丢失），再运行这一格。")

### 5.3 交付前自检

In [ ]:
import zipfile

NAME = globals().get("NAME", "你的姓名")
SUBJECT = globals().get("SUBJECT", "RM第四次培训-班级-姓名")
pdf = globals().get("pdf", ROOT / f"{SUBJECT}.pdf")

checks = [
    ("权重存在", (ROOT / f"{NAME}.pt").exists()),
    ("PDF 存在", pdf.exists()),
    ("压缩包存在", (ROOT / f"{SUBJECT}.zip").exists()),
]
if (ROOT / f"{SUBJECT}.zip").exists():
    names = zipfile.ZipFile(ROOT / f"{SUBJECT}.zip").namelist()
    checks.append(("压缩包只有 2 个文件（PDF + 权重）", len(names) == 2))

print("交付前自检：")
for name, ok in checks:
    print(f"  {'✅' if ok else '❌'} {name}")

print("""
还要人工确认（脚本查不了）：
  - 文档里每个数字都能在 runs/ 下的 results.csv / summary.csv 里查到
  - 文档写明了推荐推理配置（imgsz / conf / iou）
  - PDF 里的训练曲线、混淆矩阵等图片显示正常
  - 训练表格至少 2~3 行（baseline + 至少一组对照）
  - 收件人地址正确
""")

## 6. 排错速查

**通用顺序：先看日志 → 再对比配置 → 再看资源 → 缩小规模复现 → 最后才调参。**

| 现象 | 原因 | 解决 |
|---|---|---|
| `torch.cuda.is_available()` 为 False | CPU 版 torch / CUDA 版本不匹配 | 用官方 index-url 重装对应 cu 版本 |
| 训练 OOM | batch 太大 | 降 `--batch`（960 用 4、1280 用 2）或降 `--imgsz` |
| **训练正常、推理却 OOM** | 训练用 AMP 半精度，推理默认 FP32 | 推理降 batch，或开半精度 |
| 机器卡死、键鼠无响应 | `workers` 过大 + 内存耗尽且无 swap | workers 降到 2、限制线程数、加 6G swap |
| 比预期慢一倍（笔记本） | 显卡功耗被锁（如 50W / 上限 115W） | `Fn+Q` 切性能模式；`nvidia-smi -pl 115` |
| 续训后曲线突然断裂 | 续训时参数被覆盖（例如 imgsz） | 对比 `args.yaml` 与断点；脚本已内置护栏 |
| `Overriding model.yaml nc=80 with nc=1` | 把 COCO 的 80 类头换成 1 类 | 正常，不是错误 |
| mAP 一直很低（<0.3） | 数据/标签/类别数问题 | 依次查配对 → 坐标范围 → `nc` |
| 同一权重换尺寸评估分数对不上 | 推理尺寸与训练尺寸不一致 | 保持一致（本次模型用 960） |
| val 好但 test 差很多 | 划分有偏，或用 test 调过参 | 检查分层划分；test 只用一次 |

完整手册见 [07-排错手册](../07-排错手册.md)。

## 检查点

- [ ] 能根据"mAP50 高、mAP50-95 低"判断瓶颈并给出动作
- [ ] 会测单轮耗时并据此决定 epoch 数
- [ ] 至少完成一组单变量对照实验（本次 640 → 960）
- [ ] 训练表格已自动生成，可直接粘进文档
- [ ] 权重按 `<姓名>.pt` 命名
- [ ] 压缩包名 = 邮件主题 = PDF 文件名
- [ ] 交付前自检全部通过

到这里作业就完成了，祝顺利 🎉